# Peninjauan sumber Google Trends

Notebook ini mengelompokkan **teks yang persis sama** dari CSV gabungan, menyimpan seluruh asalnya, dan membantu keputusan manual sebelum sintesis/PAA. Kemiripan semantik belum digabungkan.

Pilih kernel Python dari `venv` proyek. Jalankan sel secara berurutan. Tidak ada API/LLM yang dipanggil. Saat pertama kali dijalankan, semua sumber berstatus `unreviewed`. Keputusan tersimpan dibaca kembali pada sesi berikutnya.

CSV gabungan harus sudah tersedia dari `src/merge_trends.py`. Bekukan enam CSV sumber selama peninjauan agar identitas sumber tetap konsisten.

In [ ]:
from pathlib import Path
import sys
import json
from collections import Counter
from html import escape
from IPython.display import HTML, display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'src' / 'review_trends.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Buka notebook dari folder proyek atau notebooks.')
sys.path.insert(0, str(ROOT / 'src'))
from review_trends import load_review, apply_decisions, save_review

SOURCE_PATH = ROOT / 'data/interim/trends_sources.csv'
REVIEW_PATH = ROOT / 'data/manual/trends_review.csv'
EXPORT_DIR = ROOT / 'data/interim/review'
print('Proyek:', ROOT)
print('Python:', sys.executable)

## Muat sumber dan keputusan sebelumnya

`topic_id` berasal dari hash teks asli, bukan urutan tampil. `research_domain` awal mengikuti nama file jika hanya ada satu domain. Ini masih perlu diverifikasi. Untuk teks lintas domain, pilih satu domain penelitian yang sesuai sebelum mengirimkannya ke sintesis atau PAA.

Daftar ID dan file asal disimpan sebagai array JSON dalam kolom CSV. Teks sumber tidak diubah atau diterjemahkan pada tahap ini.

In [ ]:
reviews = load_review(SOURCE_PATH, REVIEW_PATH)
print('Jumlah baris sumber:', sum(int(row['occurrences']) for row in reviews))
print('Jumlah teks unik:', len(reviews))
print('Status:', dict(Counter(row['status'] for row in reviews)))
print('Domain penelitian (sementara):', dict(Counter(row['research_domain'] or 'lintas domain' for row in reviews)))

## Tampilkan sumber per halaman

Ubah filter dan jalankan ulang sel berikut untuk meninjau kelompok lain. `DOMAIN` memfilter kategori asal; `STATUS` memfilter keputusan. Gunakan `None` untuk menampilkan semua. `START` dimulai dari 0. Salin `topic_id` untuk mengisi keputusan pada sel selanjutnya.

In [ ]:
DOMAIN = None  # 'kesehatan', 'keuangan', 'teknologi', atau None
STATUS = None  # atau None
KEYWORD = ''
START = 0
PAGE_SIZE = 20

visible = [row for row in reviews
           if (DOMAIN is None or DOMAIN in json.loads(row['source_domains']))
           and (STATUS is None or row['status'] == STATUS)
           and KEYWORD.casefold() in row['source_text'].casefold()]
columns = ['topic_id', 'source_text', 'source_domains', 'source_lists',
           'occurrences', 'research_domain', 'status', 'reason', 'language', 'intent']
page = visible[START:START + PAGE_SIZE]
header = ''.join('<th>' + escape(col) + '</th>' for col in columns)
body = ''.join('<tr>' + ''.join('<td>' + escape(str(row[col])) + '</td>'
                               for col in columns) + '</tr>' for row in page)
display(HTML('<div style="overflow-x:auto"><table><thead><tr>' + header +
             '</tr></thead><tbody>' + body + '</tbody></table></div>'))
print(f'{len(visible)} sumber sesuai filter; menampilkan {len(page)} mulai indeks {START}.')

## Isi keputusan manual

Blok berikut sudah dilengkapi berdasarkan arahan peneliti tanggal 10 September 2026. Setiap entri diberi komentar teks sumber sehingga dapat dicari dan diedit.

- Kesehatan: Top menjadi `needs_paa`; Rising menjadi `ready_for_synthesis`.
- `icd 10` menjadi `needs_paa` setelah maknanya diverifikasi melalui WHO dan disetujui peneliti. Seluruh sumber setelahnya sampai `abu vulkanik` serta `best sunscreen for face` juga menjadi `needs_paa`.
- `campak` muncul di Top dan Rising; keputusan rentang/Top didahulukan sehingga menjadi `needs_paa`.
- Keuangan dan teknologi: seluruh sumber menjadi `needs_paa`.
- Keputusan bantal dan alasan manual sebelumnya yang sesuai arahan dipertahankan.

Status ini mencatat keputusan peneliti, bukan hasil LLM Judgment. Top/Rising adalah asal daftar, bukan skor kualitas. Bahasa yang belum dapat ditetapkan dengan yakin diberi `unknown`; periksa sebelum sintesis. Intent PAA dikosongkan agar tidak menambahkan kebutuhan informasi sebelum pertanyaan sumber diperoleh.

Untuk mengubah keputusan, cari teks sumber pada komentar, lalu edit `status`, `reason`, `language`, `research_domain`, atau `intent`. Jangan ubah ID. Jalankan sel keputusan, kemudian sel simpan. Menjalankan ulang menerapkan isi dictionary terbaru; jika mengedit CSV secara terpisah, selaraskan perubahan dengan dictionary ini agar tidak tertimpa.

Pilihan status: `unreviewed`, `ready_for_synthesis`, `needs_paa`, `needs_review`, `excluded`. Alasan wajib untuk sumber yang ditinjau; domain dan bahasa wajib untuk siap sintesis/PAA, serta intent wajib untuk siap sintesis.

Kueri GTK dikeluarkan dari sumber aktif pada 10 September 2026. Jejak keputusan dan referensi sumber lama disimpan di `data/manual/gtk_exclusions.csv`; daftar per status di notebook hanya mencakup sumber aktif.


In [ ]:
# Keputusan peneliti; diperbarui setelah pengeluaran GTK tanggal 10 September 2026.
decisions = {
    # kesehatan | icd 10
    "topic_d92628a49bc281d1": {
        "status": "needs_paa",
        "reason": "ICD-10 merupakan klasifikasi penyakit dan masalah kesehatan dari WHO. Topik relevan, tetapi kebutuhan informasi belum spesifik sehingga perlu ditelusuri melalui PAA. Referensi: https://icd.who.int/browse10/2019/en",
        "language": "unknown",
        "research_domain": "kesehatan",
        "intent": ""
    },
    # kesehatan | jantung
    "topic_b477da794ae11c5a": {
        "status": "needs_paa",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Top dan rentang setelah icd 10 sampai abu vulkanik, kebutuhan informasi diperjelas melalui PAA.",
        "language": "id",
        "research_domain": "kesehatan",
        "intent": ""
    },
    # kesehatan | obat batuk
    "topic_589083b2c06cd948": {
        "status": "needs_paa",
        "reason": "Belum jelas intentnya secara spesifik",
        "language": "id",
        "research_domain": "kesehatan",
        "intent": ""
    },
    # kesehatan | paracetamol
    "topic_54238e9415df5dff": {
        "status": "needs_paa",
        "reason": "Belum jelas intentnya secara spesifik melalui PAA",
        "language": "unknown",
        "research_domain": "kesehatan",
        "intent": ""
    },
    # kesehatan | obat sakit gigi
    "topic_7f22f36a131e3702": {
        "status": "needs_paa",
        "reason": "Belum jelas intentnya secara spesifik melalui PAA",
        "language": "id",
        "research_domain": "kesehatan",
        "intent": ""
    },
    # kesehatan | kunyit
    "topic_1c41f6ddae5f2575": {
        "status": "needs_paa",
        "reason": "Belum jelas intentnya secara spesifik melalui PAA",
        "language": "id",
        "research_domain": "kesehatan",
        "intent": ""
    },
    # kesehatan | kista
    "topic_9872adb577a7de8e": {
        "status": "needs_paa",
        "reason": "Belum jelas intentnya secara spesifik melalui PAA",
        "language": "id",
        "research_domain": "kesehatan",
        "intent": ""
    },
    # kesehatan | satu sehat
    "topic_700d7c78a7988a8e": {
        "status": "needs_paa",
        "reason": "Belum jelas intentnya secara spesifik melalui PAA",
        "language": "id",
        "research_domain": "kesehatan",
        "intent": ""
    },
    # kesehatan | obat mata
    "topic_16ad15c2570db36d": {
        "status": "needs_paa",
        "reason": "Belum jelas intentnya secara spesifik melalui PAA",
        "language": "id",
        "research_domain": "kesehatan",
        "intent": ""
    },
    # kesehatan | hiv
    "topic_dca3d101447ca58a": {
        "status": "needs_paa",
        "reason": "Belum jelas intentnya secara spesifik melalui PAA",
        "language": "id",
        "research_domain": "kesehatan",
        "intent": ""
    },
    # kesehatan | obat flu
    "topic_5df8e76ae6f0b3b5": {
        "status": "needs_paa",
        "reason": "Belum jelas intentnya secara spesifik melalui PAA",
        "language": "id",
        "research_domain": "kesehatan",
        "intent": ""
    },
    # kesehatan | sunscreen
    "topic_c07dcf016acb1812": {
        "status": "needs_paa",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Top dan rentang setelah icd 10 sampai abu vulkanik, kebutuhan informasi diperjelas melalui PAA.",
        "language": "unknown",
        "research_domain": "kesehatan",
        "intent": ""
    },
    # kesehatan | campak
    "topic_eeb1af29f57e66dd": {
        "status": "needs_paa",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Top dan rentang setelah icd 10 sampai abu vulkanik, kebutuhan informasi diperjelas melalui PAA.",
        "language": "id",
        "research_domain": "kesehatan",
        "intent": ""
    },
    # kesehatan | ibuprofen
    "topic_a7f187a6f4f060d5": {
        "status": "needs_paa",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Top dan rentang setelah icd 10 sampai abu vulkanik, kebutuhan informasi diperjelas melalui PAA.",
        "language": "unknown",
        "research_domain": "kesehatan",
        "intent": ""
    },
    # kesehatan | methylprednisolone
    "topic_3792703dc0b1c421": {
        "status": "needs_paa",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Top dan rentang setelah icd 10 sampai abu vulkanik, kebutuhan informasi diperjelas melalui PAA.",
        "language": "unknown",
        "research_domain": "kesehatan",
        "intent": ""
    },
    # kesehatan | cetirizine
    "topic_fb8349206163fc47": {
        "status": "needs_paa",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Top dan rentang setelah icd 10 sampai abu vulkanik, kebutuhan informasi diperjelas melalui PAA.",
        "language": "unknown",
        "research_domain": "kesehatan",
        "intent": ""
    },
    # kesehatan | herpes
    "topic_44f20914e41c9570": {
        "status": "needs_paa",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Top dan rentang setelah icd 10 sampai abu vulkanik, kebutuhan informasi diperjelas melalui PAA.",
        "language": "unknown",
        "research_domain": "kesehatan",
        "intent": ""
    },
    # kesehatan | vertigo
    "topic_e5a559c8ce04fb73": {
        "status": "needs_paa",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Top dan rentang setelah icd 10 sampai abu vulkanik, kebutuhan informasi diperjelas melalui PAA.",
        "language": "unknown",
        "research_domain": "kesehatan",
        "intent": ""
    },
    # kesehatan | omeprazole
    "topic_798c6fe77221b475": {
        "status": "needs_paa",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Top dan rentang setelah icd 10 sampai abu vulkanik, kebutuhan informasi diperjelas melalui PAA.",
        "language": "unknown",
        "research_domain": "kesehatan",
        "intent": ""
    },
    # kesehatan | obat sariawan
    "topic_3d30efebcebc9b08": {
        "status": "needs_paa",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Top dan rentang setelah icd 10 sampai abu vulkanik, kebutuhan informasi diperjelas melalui PAA.",
        "language": "id",
        "research_domain": "kesehatan",
        "intent": ""
    },
    # kesehatan | obat diare
    "topic_dd93db486af14d0f": {
        "status": "needs_paa",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Top dan rentang setelah icd 10 sampai abu vulkanik, kebutuhan informasi diperjelas melalui PAA.",
        "language": "id",
        "research_domain": "kesehatan",
        "intent": ""
    },
    # kesehatan | obat sakit perut
    "topic_f412e41e2424a5d1": {
        "status": "needs_paa",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Top dan rentang setelah icd 10 sampai abu vulkanik, kebutuhan informasi diperjelas melalui PAA.",
        "language": "id",
        "research_domain": "kesehatan",
        "intent": ""
    },
    # kesehatan | obat asam lambung
    "topic_fab3cefa5cb88db1": {
        "status": "needs_paa",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Top dan rentang setelah icd 10 sampai abu vulkanik, kebutuhan informasi diperjelas melalui PAA.",
        "language": "id",
        "research_domain": "kesehatan",
        "intent": ""
    },
    # kesehatan | cefixime
    "topic_1b4ac6feb0601661": {
        "status": "needs_paa",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Top dan rentang setelah icd 10 sampai abu vulkanik, kebutuhan informasi diperjelas melalui PAA.",
        "language": "unknown",
        "research_domain": "kesehatan",
        "intent": ""
    },
    # kesehatan | pcare
    "topic_f9e9e55b78ed2e12": {
        "status": "needs_paa",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Top dan rentang setelah icd 10 sampai abu vulkanik, kebutuhan informasi diperjelas melalui PAA.",
        "language": "unknown",
        "research_domain": "kesehatan",
        "intent": ""
    },
    # kesehatan | obat batuk berdahak
    "topic_057246d1d024b9dc": {
        "status": "needs_paa",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Top dan rentang setelah icd 10 sampai abu vulkanik, kebutuhan informasi diperjelas melalui PAA.",
        "language": "id",
        "research_domain": "kesehatan",
        "intent": ""
    },
    # kesehatan | vitamin c
    "topic_d4ebb3df9eed1d5f": {
        "status": "needs_paa",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Top dan rentang setelah icd 10 sampai abu vulkanik, kebutuhan informasi diperjelas melalui PAA.",
        "language": "unknown",
        "research_domain": "kesehatan",
        "intent": ""
    },
    # kesehatan | obat asam urat
    "topic_cded8492075aeb07": {
        "status": "needs_paa",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Top dan rentang setelah icd 10 sampai abu vulkanik, kebutuhan informasi diperjelas melalui PAA.",
        "language": "id",
        "research_domain": "kesehatan",
        "intent": ""
    },
    # kesehatan | pneumonia
    "topic_72a8df6694d5ca04": {
        "status": "needs_paa",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Top dan rentang setelah icd 10 sampai abu vulkanik, kebutuhan informasi diperjelas melalui PAA.",
        "language": "unknown",
        "research_domain": "kesehatan",
        "intent": ""
    },
    # kesehatan | abu vulkanik
    "topic_993c5d46a8798752": {
        "status": "needs_paa",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Top dan rentang setelah icd 10 sampai abu vulkanik, kebutuhan informasi diperjelas melalui PAA.",
        "language": "id",
        "research_domain": "kesehatan",
        "intent": ""
    },
    # kesehatan | best sunscreen for face
    "topic_79f8cad508dfd24c": {
        "status": "needs_paa",
        "reason": "Sesuai keputusan peneliti, telusuri PAA untuk memperjelas kebutuhan sunscreen, misalnya terkait jenis kulit; jangan menambahkan jenis kulit tanpa sumber.",
        "language": "en",
        "research_domain": "kesehatan",
        "intent": ""
    },
    # kesehatan | how to treat seasonal allergies
    "topic_b1bb11fc1e5a5953": {
        "status": "ready_for_synthesis",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Rising, kebutuhan informasi dipertahankan dan dirumuskan dalam bahasa Indonesia tanpa menambah konteks.",
        "language": "en",
        "research_domain": "kesehatan",
        "intent": "Mencari cara menangani alergi musiman."
    },
    # kesehatan | how to sleep better at night
    "topic_79d4626803fbf030": {
        "status": "ready_for_synthesis",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Rising, kebutuhan informasi dipertahankan dan dirumuskan dalam bahasa Indonesia tanpa menambah konteks.",
        "language": "en",
        "research_domain": "kesehatan",
        "intent": "Mencari cara tidur lebih nyenyak pada malam hari."
    },
    # kesehatan | anxiety coping strategies
    "topic_12ca4524eda8479d": {
        "status": "ready_for_synthesis",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Rising, kebutuhan informasi dipertahankan dan dirumuskan dalam bahasa Indonesia tanpa menambah konteks.",
        "language": "en",
        "research_domain": "kesehatan",
        "intent": "Mencari strategi menghadapi kecemasan."
    },
    # kesehatan | best posture correctors
    "topic_4387e1bf6a23c97e": {
        "status": "ready_for_synthesis",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Rising, kebutuhan informasi dipertahankan dan dirumuskan dalam bahasa Indonesia tanpa menambah konteks.",
        "language": "en",
        "research_domain": "kesehatan",
        "intent": "Mencari rekomendasi alat koreksi postur."
    },
    # kesehatan | best pillow for side sleepers
    "topic_f61cef5cdb4fd375": {
        "status": "ready_for_synthesis",
        "reason": "Kebutuhan informasi sudah jelas dan dapat diterjemahkan ke bahasa Indonesia tanpa menambah konteks.",
        "language": "en",
        "research_domain": "kesehatan",
        "intent": "Mencari rekomendasi bantal yang sesuai untuk orang yang tidur menyamping."
    },
    # kesehatan | symptoms of iron deficiency
    "topic_ae2bb8890994df81": {
        "status": "ready_for_synthesis",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Rising, kebutuhan informasi dipertahankan dan dirumuskan dalam bahasa Indonesia tanpa menambah konteks.",
        "language": "en",
        "research_domain": "kesehatan",
        "intent": "Mencari gejala kekurangan zat besi."
    },
    # kesehatan | symptoms of dehydration
    "topic_ef04a6782179c6fc": {
        "status": "ready_for_synthesis",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Rising, kebutuhan informasi dipertahankan dan dirumuskan dalam bahasa Indonesia tanpa menambah konteks.",
        "language": "en",
        "research_domain": "kesehatan",
        "intent": "Mencari gejala dehidrasi."
    },
    # kesehatan | symptoms of the flu
    "topic_68a8bdd5a54a6c04": {
        "status": "ready_for_synthesis",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Rising, kebutuhan informasi dipertahankan dan dirumuskan dalam bahasa Indonesia tanpa menambah konteks.",
        "language": "en",
        "research_domain": "kesehatan",
        "intent": "Mencari gejala flu."
    },
    # kesehatan | symptoms of covid vs flu
    "topic_8ed503adcbb5cbb3": {
        "status": "ready_for_synthesis",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Rising, kebutuhan informasi dipertahankan dan dirumuskan dalam bahasa Indonesia tanpa menambah konteks.",
        "language": "en",
        "research_domain": "kesehatan",
        "intent": "Mencari perbandingan gejala COVID dan flu."
    },
    # kesehatan | lactose intolerance symptoms
    "topic_53d5f4393229bec7": {
        "status": "ready_for_synthesis",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Rising, kebutuhan informasi dipertahankan dan dirumuskan dalam bahasa Indonesia tanpa menambah konteks.",
        "language": "en",
        "research_domain": "kesehatan",
        "intent": "Mencari gejala intoleransi laktosa."
    },
    # kesehatan | how to reduce stress naturally
    "topic_e82306b152e791a8": {
        "status": "ready_for_synthesis",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Rising, kebutuhan informasi dipertahankan dan dirumuskan dalam bahasa Indonesia tanpa menambah konteks.",
        "language": "en",
        "research_domain": "kesehatan",
        "intent": "Mencari cara mengurangi stres secara alami."
    },
    # kesehatan | symptoms of vitamin d deficiency
    "topic_5835e6541077d20e": {
        "status": "ready_for_synthesis",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Rising, kebutuhan informasi dipertahankan dan dirumuskan dalam bahasa Indonesia tanpa menambah konteks.",
        "language": "en",
        "research_domain": "kesehatan",
        "intent": "Mencari gejala kekurangan vitamin D."
    },
    # kesehatan | how to measure blood pressure at home
    "topic_db8e0ecc5afa88bd": {
        "status": "ready_for_synthesis",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Rising, kebutuhan informasi dipertahankan dan dirumuskan dalam bahasa Indonesia tanpa menambah konteks.",
        "language": "en",
        "research_domain": "kesehatan",
        "intent": "Mencari cara mengukur tekanan darah di rumah."
    },
    # kesehatan | symptoms of food poisoning
    "topic_c0e4cc083cbc34ba": {
        "status": "ready_for_synthesis",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Rising, kebutuhan informasi dipertahankan dan dirumuskan dalam bahasa Indonesia tanpa menambah konteks.",
        "language": "en",
        "research_domain": "kesehatan",
        "intent": "Mencari gejala keracunan makanan."
    },
    # kesehatan | best face sunscreen for men
    "topic_9d3e9e66994440d3": {
        "status": "ready_for_synthesis",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Rising, kebutuhan informasi dipertahankan dan dirumuskan dalam bahasa Indonesia tanpa menambah konteks.",
        "language": "en",
        "research_domain": "kesehatan",
        "intent": "Mencari rekomendasi tabir surya wajah untuk pria."
    },
    # kesehatan | best at home teeth whitening kits
    "topic_13e486265923d230": {
        "status": "ready_for_synthesis",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Rising, kebutuhan informasi dipertahankan dan dirumuskan dalam bahasa Indonesia tanpa menambah konteks.",
        "language": "en",
        "research_domain": "kesehatan",
        "intent": "Mencari rekomendasi perlengkapan pemutih gigi untuk digunakan di rumah."
    },
    # kesehatan | best protein bars
    "topic_dc210eedcb807a35": {
        "status": "ready_for_synthesis",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Rising, kebutuhan informasi dipertahankan dan dirumuskan dalam bahasa Indonesia tanpa menambah konteks.",
        "language": "en",
        "research_domain": "kesehatan",
        "intent": "Mencari rekomendasi protein bar."
    },
    # kesehatan | how to reduce sugar intake
    "topic_0d30fe4752921634": {
        "status": "ready_for_synthesis",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Rising, kebutuhan informasi dipertahankan dan dirumuskan dalam bahasa Indonesia tanpa menambah konteks.",
        "language": "en",
        "research_domain": "kesehatan",
        "intent": "Mencari cara mengurangi asupan gula."
    },
    # kesehatan | healthy snacks for work
    "topic_24e2ad47794e3300": {
        "status": "ready_for_synthesis",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Rising, kebutuhan informasi dipertahankan dan dirumuskan dalam bahasa Indonesia tanpa menambah konteks.",
        "language": "en",
        "research_domain": "kesehatan",
        "intent": "Mencari pilihan camilan sehat untuk bekerja."
    },
    # kesehatan | best vitamins for women
    "topic_d6ecf557614f0c11": {
        "status": "ready_for_synthesis",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Rising, kebutuhan informasi dipertahankan dan dirumuskan dalam bahasa Indonesia tanpa menambah konteks.",
        "language": "en",
        "research_domain": "kesehatan",
        "intent": "Mencari rekomendasi vitamin untuk perempuan."
    },
    # kesehatan | essential oils for sleep
    "topic_13da336f0d1800d9": {
        "status": "ready_for_synthesis",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Rising, kebutuhan informasi dipertahankan dan dirumuskan dalam bahasa Indonesia tanpa menambah konteks.",
        "language": "en",
        "research_domain": "kesehatan",
        "intent": "Mencari informasi tentang minyak esensial untuk tidur."
    },
    # kesehatan | how to lower blood pressure
    "topic_14d1df7e7dd1af76": {
        "status": "ready_for_synthesis",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Rising, kebutuhan informasi dipertahankan dan dirumuskan dalam bahasa Indonesia tanpa menambah konteks.",
        "language": "en",
        "research_domain": "kesehatan",
        "intent": "Mencari cara menurunkan tekanan darah."
    },
    # kesehatan | how to track menstrual cycle
    "topic_a15d419795e1c71f": {
        "status": "ready_for_synthesis",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Rising, kebutuhan informasi dipertahankan dan dirumuskan dalam bahasa Indonesia tanpa menambah konteks.",
        "language": "en",
        "research_domain": "kesehatan",
        "intent": "Mencari cara melacak siklus menstruasi."
    },
    # kesehatan | best kettlebell exercises
    "topic_24d534d2fb34a047": {
        "status": "ready_for_synthesis",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Rising, kebutuhan informasi dipertahankan dan dirumuskan dalam bahasa Indonesia tanpa menambah konteks.",
        "language": "en",
        "research_domain": "kesehatan",
        "intent": "Mencari rekomendasi latihan kettlebell."
    },
    # kesehatan | how to read nutrition labels
    "topic_ec102088c6ad4a58": {
        "status": "ready_for_synthesis",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Rising, kebutuhan informasi dipertahankan dan dirumuskan dalam bahasa Indonesia tanpa menambah konteks.",
        "language": "en",
        "research_domain": "kesehatan",
        "intent": "Mencari cara membaca label gizi."
    },
    # kesehatan | hakikat senam aerobik menurut jackie sorensens
    "topic_d49592330ebc1d6a": {
        "status": "ready_for_synthesis",
        "reason": "Sesuai keputusan peneliti untuk sumber kesehatan Rising, kebutuhan informasi dipertahankan dan dirumuskan dalam bahasa Indonesia tanpa menambah konteks.",
        "language": "id",
        "research_domain": "kesehatan",
        "intent": "Mencari pengertian senam aerobik menurut Jackie Sorensens."
    },
    # keuangan | rupiah
    "topic_95e9dadbbf2f9c84": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "id",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | idr
    "topic_a65110e3d8c4150d": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | bank
    "topic_4381dc2ab1428516": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "id",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | pajak
    "topic_9ed5b4fe6af73d5c": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "id",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | dolar
    "topic_2db2327f595f8e9d": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "id",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | usd
    "topic_d60fbd8718eceb7f": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | bca
    "topic_96e31900cc68fc88": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | uang
    "topic_f23f3a54a30ba8a5": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "id",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | usd idr
    "topic_b0344bd53faf4464": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | usd to idr
    "topic_fb6e402a531dc07b": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | idr to usd
    "topic_460aaa73178dc5a2": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | emas hari ini
    "topic_e4feaea26deaf551": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | harga emas hari ini
    "topic_80e83320683a0299": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | coretax
    "topic_1ec660f0d5fd7134": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | bni
    "topic_78f88acaa89ae95c": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | npwp
    "topic_c9925199989e0b36": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | klikbca
    "topic_5af00eba37b032aa": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | djp
    "topic_df76040a115fce1b": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | dollar hari ini
    "topic_97149db96fa8f2ff": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | 1 dolar
    "topic_62f7de007ff23ed3": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | djp coretax
    "topic_74c6a3d4b71c54fe": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | coretax djp
    "topic_09c0e4c3b5c62c44": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | dollar ke rupiah
    "topic_657059d70170dad0": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | 1 dolar berapa rupiah
    "topic_2ebaa409d1271717": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | rupiah hari ini
    "topic_47b815b25e4587ab": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | bpjs ketenagakerjaan
    "topic_2e9eff06d1a8a618": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | pembayaran
    "topic_738cece1a32aecde": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "id",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | pegadaian
    "topic_bdb492f26307f774": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "id",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | dolar ke rupiah
    "topic_9c1e164d854055a1": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | bank mandiri
    "topic_62d13350ab6ce3f3": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | rupiah ke dollar
    "topic_29d37e4a2eb8146d": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | kurs dollar
    "topic_68e3b311b7efd0c2": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | bpjs kesehatan
    "topic_9fc31d4ce7cfa44d": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | dolar hari ini
    "topic_433d80b9fd77840c": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | bank indonesia
    "topic_54ea518ccc5f2aef": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | btn
    "topic_2dbdc96b9117837a": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | npwp online
    "topic_3a0493e4111f466a": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | antam hari ini
    "topic_46a0128899fca0f1": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | sgd idr
    "topic_d4665896e41b8837": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | subsidi tepat
    "topic_052bda05d5e5d8c2": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | coretax pajak
    "topic_4219364107dea791": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | mandiri kopra
    "topic_c56b756b99bbd82f": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | kopra mandiri
    "topic_673b878b6818c982": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | sgd to idr
    "topic_bb59b35cea8995d8": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | klik bca
    "topic_7b9520018b33bca4": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | lynk
    "topic_2ed2a55e12ae4ff3": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | kur bri 2026
    "topic_df50651a8a38540e": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | bos dolar
    "topic_a4dfbfecbcafba15": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | pinjaman daring
    "topic_a5614b5987d9c9fb": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | harga perak hari ini
    "topic_183502216e7d4b1c": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | bank central asia
    "topic_36a4606aa006e787": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | harga bbm hari ini
    "topic_5d77521d4384165c": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | bank negara indonesia
    "topic_dd7a081c94f40aa2": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | harga emas dunia hari ini
    "topic_9f259c62d24ad58a": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | goldprice
    "topic_3b3821dcad0b71c8": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "en",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | harga pertamax hari ini
    "topic_718ff269906c2f3b": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | pintasan atr bpn
    "topic_0749df3adf15f37d": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | harga emas galeri 24 hari ini
    "topic_24b4c6a511fe0b81": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | ihsg
    "topic_845192799de9d38f": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | ihsg hari ini
    "topic_1e5e359dba8e69a1": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | investor
    "topic_df22016bdd63f389": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "en",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | cortex pajak
    "topic_9d7e3b3eb1c7ac9c": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | gold price
    "topic_7d985056e57f34f5": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "en",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | 1 dolar berapa rupiah hari ini
    "topic_99c5279b40e049d2": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | harga emas perhiasan hari ini
    "topic_7a90040cc0da8fb6": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | coretaxdjp
    "topic_34f00c1ee9b71a6c": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | dividen
    "topic_7b5ae63a44dc58a0": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "id",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | cny to idr
    "topic_c552653aac3a58a4": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | kurs dolar hari ini
    "topic_df838f85d4af19ea": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | aud to idr
    "topic_ffd78f6fd2244153": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | harga dolar hari ini
    "topic_e37c8e295443ac2e": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | trading view
    "topic_e49bd04de8f9239e": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "en",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | yuan to idr
    "topic_4187c984313b940c": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | harga emas antam hari ini
    "topic_0272ef623ab215e2": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | harga antam hari ini
    "topic_f841dbf295358b2b": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # keuangan | emas antam hari ini
    "topic_ab98dd2a49c5a81a": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain keuangan; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "keuangan",
        "intent": ""
    },
    # teknologi | word
    "topic_98c1eb4ee9347674": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "en",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | aplikasi
    "topic_4055d6640423572e": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "id",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | pdf to word
    "topic_0b99f4f3970c4844": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "en",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | word to pdf
    "topic_53d71eb4f936e726": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "en",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | google
    "topic_bbdefa2950f49882": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | oppo
    "topic_8147e6fed86d42c1": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | wa
    "topic_87b810070b7c7396": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | vpn
    "topic_ebf20cefc9169e0b": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | font
    "topic_795ea3efa43d0872": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "en",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | excel
    "topic_faba1e00af8e6c89": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "en",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | croxy proxy
    "topic_a539d08cf120c247": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | drive
    "topic_7062520c5a0ea9de": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "en",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | youtube
    "topic_24e6654bfd1ab85b": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | driver
    "topic_b4def8217cadae26": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "en",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | chrome
    "topic_9390ef32addf32bf": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | ig
    "topic_26a3482d50cfc162": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | whatsapp
    "topic_ec8202b6f9fb16f9": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | gemini
    "topic_5d72436256ada538": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | asn digital
    "topic_19b44fb002e45dc1": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | proxy vpn
    "topic_9d492bad4d2f9610": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "en",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | mp4
    "topic_862c4ec62defaada": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | github
    "topic_c0b0109d9439de57": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | wa web
    "topic_7ef888fb88cb6cd5": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | proxi
    "topic_6a04ecb8afdffbd9": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | blackbox
    "topic_12708c45a93e4d6e": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | spreadsheet
    "topic_53e54fd268849dff": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "en",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | yt
    "topic_3d40f99acd3b0e84": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | duckduckgo
    "topic_5b8eeb90771c2031": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | convert pdf to word
    "topic_39d13ae3af52aa09": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "en",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | microsoft
    "topic_9fbf261b62c1d7c0": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | blackbox ai
    "topic_33a010f94e7aab4e": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | convert word to pdf
    "topic_9dd87bba4367923f": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "en",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | docs
    "topic_46b42b4229cd7a39": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "en",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | fb
    "topic_5f3731b52ad0fb2f": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | compress pdf
    "topic_6eaaccbf3f5b566b": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "en",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | qr
    "topic_d847acf7bab1b6f7": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | chatgpt
    "topic_60965168ce762e94": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | translate
    "topic_7fd690e20d081646": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | gmail
    "topic_576ba7c2e4abb718": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | camera
    "topic_89e8b9518d922794": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "en",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | ibox
    "topic_25314728e6e4e63b": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | gpt
    "topic_053ea4804ef1bb33": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | merge pdf
    "topic_075c20b3ae29f596": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "en",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | pdf to excel
    "topic_fac4fe2dc2a4efe5": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "en",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | canva
    "topic_3db1f560db626918": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | excel to pdf
    "topic_4d6a3903005260fb": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "en",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | facebook
    "topic_3d59f7548e1af215": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | aplikasi cek bansos
    "topic_cd51f8c172664759": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | proxy poxy
    "topic_7b4f83b295adacf6": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | my asn digital
    "topic_8238323b671b543d": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | maxstream
    "topic_3a74b2977dbcfe5b": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | gemini ai
    "topic_e8dd817327bd07e8": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | prxy
    "topic_f2305b58b6760b7f": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | proxy browser
    "topic_e3f0bdf8dcb795d6": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "en",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | proxy server
    "topic_25d9a403ad166022": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "en",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | prox
    "topic_d187c967da6fb7b6": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | proxy site vpn
    "topic_fd2b82b08c8c2983": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "en",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | proxsy
    "topic_224b3758324b6dea": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | proky
    "topic_e523a65715e9c36f": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | lacak hp gmail
    "topic_ea637f597071756a": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | font gratis
    "topic_4d468a253d7e7e8e": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | free fonts
    "topic_02e395dfb76fce67": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "en",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | outlook
    "topic_4f53c6c301f98610": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | cai
    "topic_61067a1f3a73ef18": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | i love pdf to word
    "topic_7ae312c7453f46a6": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | gform
    "topic_b010150744b8d519": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | merger pdf
    "topic_4f5e1741190eabb3": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | ilovepdf
    "topic_6a1492e15aff00ff": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | ekinerja
    "topic_39a700b35ba80494": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | accurate
    "topic_10cdb6d6fe701fe6": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | i love pdf
    "topic_1f003b10e0e01d87": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | icloud login
    "topic_5a83f05991e88f2d": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | laptop asus
    "topic_c99910eca7614b65": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | foto to pdf
    "topic_2d73fdf14e1be4c2": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | accurate online
    "topic_5a79be092b735d03": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "unknown",
        "research_domain": "teknologi",
        "intent": ""
    },
    # teknologi | spreadsheets
    "topic_eb2bfd49b36e1ae4": {
        "status": "needs_paa",
        "reason": "Diarahkan ke pengumpulan PAA sesuai keputusan peneliti untuk seluruh sumber domain teknologi; intent tidak ditambahkan pada tahap ini.",
        "language": "en",
        "research_domain": "teknologi",
        "intent": ""
    },
}

reviews = apply_decisions(reviews, decisions)
print("Keputusan diterapkan di memori. Jalankan sel berikut untuk menyimpan.")


## Simpan progres dan ekspor daftar kerja

Sel ini menyimpan `data/manual/trends_review.csv` dan lima CSV per status di `data/interim/review/`. Ekspor selalu mengikuti keputusan terkini; sumber belum ditinjau tidak masuk daftar siap sintesis atau PAA. Berkas hasil sebelumnya diganti saat menyimpan.

Anda dapat berhenti dan melanjutkan notebook nanti. Untuk sesi berikutnya, muat kembali sumber dan keputusan dari atas. Jika mengedit CSV keputusan dengan spreadsheet, tutup/simpan berkas lalu muat ulang sebelum melanjutkan. Hindari mengedit CSV dan notebook bersamaan.

In [ ]:
summary = save_review(reviews, REVIEW_PATH, EXPORT_DIR)
print('Keputusan tersimpan:', REVIEW_PATH)
print('Ringkasan:', summary)
print('Daftar kerja:', EXPORT_DIR)
remaining = sum(row['status'] in {'unreviewed', 'needs_review'} for row in reviews)
print('Masih perlu peninjauan:', remaining)

## Setelah peninjauan

- `ready_for_synthesis.csv`: masukan uji prompt setelah Judgment pertama.
- `needs_paa.csv`: daftar topik untuk pengambilan PAA, dengan hubungan ke sumber Trends.
- `needs_review.csv`, `unreviewed.csv`, dan `excluded.csv`: catatan seleksi yang tetap dipertahankan.

Jangan memaksakan semua topik menjadi pertanyaan. Periksa pula dominasi topik/intent, aturan penerjemahan, dan cakupan domain sebelum memilih batch uji.